# Analysis of Trending NFTs using Time-Series Data and Machine Learning
**Pipeline:** Data → Cleaning → Feature Engineering → PCA → SVM / Random Forest → Evaluation

Run each cell top-to-bottom (`Shift+Enter`). Runtime: CPU is fine; for faster SVM use *Runtime → Change runtime type → High-RAM* if available.

## 1. Setup — install & import libraries

In [ ]:
!pip -q install scikit-learn pandas numpy matplotlib seaborn joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, time, json

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Setup complete")

## 2. Load the dataset

This project uses the real **FinWeb / TweetBoost dataset** (62,997 OpenSea assets with
Twitter + OpenSea features), already merged into one file: `nft_dataset_real.csv` (~40 MB).

**Upload it into this Colab session:** click the folder icon in the left sidebar,
then the upload button (page with an up-arrow), and pick `nft_dataset_real.csv`.
Wait for the orange progress circle to finish before running the next cell.

If you run this cell *without* uploading first, a file-picker dialog opens —
choose the CSV there instead. (Note: session uploads are deleted when Colab
disconnects, so you will need to re-upload next session.)

In [ ]:
import os

DATA_PATH = 'nft_dataset_real.csv'
GENERATE_SYNTHETIC = False  # set True only if you don't have the real CSV

if not GENERATE_SYNTHETIC and not os.path.exists(DATA_PATH):
    # File not found in the session -> open the upload dialog.
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]

if GENERATE_SYNTHETIC:
    DATA_PATH = 'nft_dataset.csv'

In [ ]:
if GENERATE_SYNTHETIC:
    rng = np.random.default_rng(RANDOM_STATE)
    rows = 70_000
    hype = rng.normal(0, 1, rows)
    price = np.clip(np.exp((1.2 + 1.9*hype + rng.normal(0, 1.1, rows)) * 0.9), 0.01, 5e5)
    df = pd.DataFrame({
        'category': rng.choice(['art','collectible','games','metaverse','music','sports','utility'], rows),
        'followers': np.exp(6 + 1.4*hype + rng.normal(0,1.5,rows)).astype(int),
        'followings': np.exp(5 + 0.3*hype + rng.normal(0,1.2,rows)).astype(int),
        'tweet_likes': np.exp(1.5 + 1.1*hype + rng.normal(0,1.3,rows)).astype(int),
        'tweet_count': rng.poisson(np.clip(2 + 2.5*np.exp(0.5*hype), 0, 60)),
        'account_age_days': rng.integers(30, 4000, rows),
        'verified': (rng.random(rows) < 1/(1+np.exp(-(hype-2)))).astype(int),
        'num_sales': rng.poisson(np.clip(0.3 + 1.5*np.exp(0.6*hype), 0, 300)),
        'listed_count': rng.poisson(np.clip(1 + 2.0*np.exp(0.4*hype), 0, 500)),
        'collection_size': np.exp(rng.normal(7, 2, rows)).astype(int),
        'days_since_mint': rng.integers(1, 365, rows),
        'favorites_on_opensea': np.exp(0.5 + 0.9*hype + rng.normal(0,1.0,rows)).astype(int),
        'rarity_score': np.clip(rng.normal(50 + 8*hype, 20, rows), 0, 100),
        'avg_price_usd': np.round(price, 2),
    })
    df['retweets'] = (df['tweet_likes'] * rng.uniform(0.05, 0.4, rows)).astype(int)
    # inject missing values like real scraped data
    for c in ['tweet_likes','retweets','followers','rarity_score']:
        df.loc[rng.random(rows) < 0.03, c] = np.nan
    df.to_csv(DATA_PATH, index=False)

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print(df.info())
print('\nMissing values per column:')
print(df.isna().sum()[df.isna().sum() > 0])
df.describe().T

In [ ]:
sold = df.loc[df['avg_price_usd'] > 0, 'avg_price_usd']
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(np.log10(sold), bins=80, color='#4c72b0')
axes[0].set_title(f'Price distribution of sold assets (n={len(sold):,}, log10 USD)')
axes[0].set_xlabel('log10(price)')

share = [(df['avg_price_usd'] == 0).mean(), (df['avg_price_usd'] > 1500).mean()]
axes[1].bar(['never sold', '> $1500'], share, color=['#55a868', '#c44e52'])
axes[1].set_title('Market skew (report: ~76% never sold, ~1% premium)')
axes[1].set_ylabel('share of assets')
plt.tight_layout(); plt.show()

In [ ]:
num = df.select_dtypes(include=[np.number])
corr = num.corr()['avg_price_usd'].drop('avg_price_usd').abs().sort_values(ascending=False)
top = list(corr.head(12).index)
plt.figure(figsize=(10, 8))
sns.heatmap(num[top + ['avg_price_usd']].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation of top features with price'); plt.tight_layout(); plt.show()

## 4. Pre-processing

1. **Clean**: drop duplicates.
2. **Feature engineering**: log-transform skewed counts, build engagement ratios
   (works with both the real FinWeb schema and the synthetic one).
3. **Target**: the real dataset ships `price_label` — 0 = never sold, 1..5 =
   logarithmic price bins (the `asset valuation` approach, since most NFTs have
   too few sales for direct price regression). For synthetic data we bin
   `avg_price_usd` ourselves.

In [ ]:
df = df.drop_duplicates()

LOG_COLS = [c for c in ['followers','followings','tweet_likes','retweets',
                        'collection_size','favorites_on_opensea',
                        'followers_count_mean','friends_count_mean','nlikes_mean',
                        'nretweets_mean','nreplies_mean','favourites_count_mean',
                        'statuses_count_mean','total_volume','total_sales',
                        'total_supply','market_cap','num_owners'] if c in df.columns]
for c in LOG_COLS:
    df[f'log_{c}'] = np.log1p(df[c].clip(lower=0))

def has(*cols): return all(c in df.columns for c in cols)
if has('nlikes_mean','nretweets_mean','followers_count_mean'):  # real schema
    df['engagement_rate'] = (df['nlikes_mean'] + df['nretweets_mean']) / (df['followers_count_mean'] + 1)
    df['follower_following_ratio'] = df['followers_count_mean'] / (df['friends_count_mean'] + 1)
elif has('tweet_likes','retweets','followers'):                  # synthetic schema
    df['engagement_rate'] = (df['tweet_likes'] + df['retweets']) / (df['followers'] + 1)
    df['follower_following_ratio'] = df['followers'] / (df['followings'] + 1)

if 'price_label' in df.columns:   # real dataset: use its ready-made target
    bad = df['price_label'].isna().sum()
    if bad:
        # NaN labels usually mean the CSV upload was truncated — check the
        # dataframe shape printed in step 2: it must be (62997, 125).
        print(f'WARNING: dropping {bad} rows with missing price_label '
              f'(if this is more than a handful, re-upload the CSV)')
        df = df.dropna(subset=['price_label'])
    names = {0:'unsold', 1:'bin1', 2:'bin2', 3:'bin3', 4:'bin4', 5:'bin5'}
    df['price_class'] = df['price_label'].astype(int).map(names)
else:                              # synthetic: bin the price ourselves
    df = df[df['avg_price_usd'] > 0].copy()
    df['price_class'] = pd.cut(df['avg_price_usd'], bins=[0,15,100,1500,np.inf],
        labels=['low(<$15)','mid($15-100)','high($100-1500)','premium(>$1500)'],
        include_lowest=True)

y = df['price_class']
DROP = ['avg_price_usd','price_class','price_label','sale_label','permalink','asset_id']
X = df.drop(columns=[c for c in DROP if c in df.columns])
print('Class balance:'); print(y.value_counts(normalize=True).round(3))

## 5. Train/test split + preprocessing pipeline (impute → scale → one-hot → PCA)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

categorical = [c for c in X.columns if X[c].dtype == 'object']
numeric = [c for c in X.columns if c not in categorical]

preprocessor = Pipeline([
    ('columns', ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), numeric),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                          ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical),
    ])),
    ('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)),  # keep 95% variance
])
print(f'Train {len(X_train):,} | Test {len(X_test):,} | Raw features {X.shape[1]}')

## 6. Model 1 — Random Forest

In [ ]:
rf_pipe = Pipeline([('prep', clone(preprocessor)),
                    ('clf', RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                                   class_weight='balanced', n_jobs=-1,
                                                   random_state=RANDOM_STATE))])
t0 = time.time(); rf_pipe.fit(X_train, y_train); print(f'fit {time.time()-t0:.1f}s')
rf_pred = rf_pipe.predict(X_test)
print(f'Random Forest accuracy: {accuracy_score(y_test, rf_pred):.4f}')
print(classification_report(y_test, rf_pred, zero_division=0))

## 7. Model 2 — SVM (RBF kernel)
SVM training is O(n²) — we subsample the training split to keep Colab runtime reasonable. Increase `SVM_SAMPLE` if you have time/RAM.

In [ ]:
SVM_SAMPLE = 15_000
if len(X_train) > SVM_SAMPLE:
    idx = np.random.default_rng(RANDOM_STATE).choice(len(X_train), SVM_SAMPLE, replace=False)
    X_svm, y_svm = X_train.iloc[idx], y_train.iloc[idx]
else:
    X_svm, y_svm = X_train, y_train

svm_pipe = Pipeline([('prep', clone(preprocessor)),
                     ('clf', SVC(kernel='rbf', C=10, gamma='scale',
                                 class_weight='balanced', random_state=RANDOM_STATE))])
t0 = time.time(); svm_pipe.fit(X_svm, y_svm); print(f'fit {time.time()-t0:.1f}s')
svm_pred = svm_pipe.predict(X_test)
print(f'SVM accuracy: {accuracy_score(y_test, svm_pred):.4f}')
print(classification_report(y_test, svm_pred, zero_division=0))

## 8. Evaluation — confusion matrices & PCA variance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (name, pred) in zip(axes, [('Random Forest', rf_pred), ('SVM (RBF)', svm_pred)]):
    cm = confusion_matrix(y_test, pred, normalize='true')
    ConfusionMatrixDisplay(cm, display_labels=sorted(y.unique())).plot(
        ax=ax, cmap='Blues', colorbar=False, xticks_rotation=30, values_format='.2f')
    ax.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
pca = rf_pipe.named_steps['prep'].named_steps['pca']
plt.figure(figsize=(8, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o', ms=3)
plt.axhline(0.95, ls='--', c='red', label='95% variance')
plt.xlabel('number of principal components'); plt.ylabel('cumulative explained variance')
plt.title(f'PCA: {pca.n_components_} components retained'); plt.legend(); plt.show()

## 9. (Optional) Hyperparameter tuning with GridSearchCV
This can take 10–30+ minutes — run it once you're happy with the base pipeline.

In [ ]:
RUN_TUNING = False
if RUN_TUNING:
    grid = GridSearchCV(
        Pipeline([('prep', clone(preprocessor)), ('clf', RandomForestClassifier(
            n_jobs=-1, class_weight='balanced', random_state=RANDOM_STATE))]),
        param_grid={'clf__n_estimators': [200, 400],
                    'clf__max_depth': [None, 20, 40],
                    'clf__min_samples_leaf': [1, 2, 5]},
        cv=3, scoring='f1_macro', n_jobs=-1, verbose=1)
    grid.fit(X_train, y_train)
    print('Best params:', grid.best_params_)
    print('Best CV f1_macro:', grid.best_score_)

## 10. Save the trained models (download or copy to Drive)

In [ ]:
joblib.dump(rf_pipe, 'random_forest_nft.joblib')
joblib.dump(svm_pipe, 'svm_nft.joblib')
print('Saved: random_forest_nft.joblib, svm_nft.joblib')

# Download to your computer:
# from google.colab import files
# files.download('random_forest_nft.joblib')

# Or copy to Drive (after mounting in step 2):
# !cp random_forest_nft.joblib /content/drive/MyDrive/

## 11. Predict on a new NFT
Load the saved model later with `joblib.load` and call `.predict()` on a DataFrame containing the **same raw feature columns** (the pipeline handles imputation/scaling/PCA automatically).

In [ ]:
sample = X_test.iloc[[0]]
print('Predicted:', rf_pipe.predict(sample)[0])
print('Actual:   ', y_test.iloc[0])